# ISIC 2024 — Stage-3 Bidirectional Co-Attention Experiment (`v2_stage3_bidir`)

Notebook thử nghiệm kiến trúc **Multi-Token Bidirectional Co-Attention Fusion** ngay tại **Stage 3**, **tái sử dụng 100%** trọng số pretrained có sẵn:
- **Stage 1 (Tabular):** `TabM` (`khoangdz/isic-2024-tabm-backbone`, OOF pAUC = `0.1682`)
- **Stage 2 (Image):** `EfficientNet-B0 @ 224px` có sẵn trong `backend/backbone_image/` (OOF pAUC = `0.1463`)
- **Mục tiêu so sánh:** Vượt mốc baseline Fusion v1 (`OOF pAUC = 0.17537`, Fold 0 = `0.17083`, Fold 1 = `0.17814`) trước khi quyết định có train lại Stage 2 với Vision Backbone lớn hơn hay nâng cấp Web Demo.

In [ ]:
# 1. Kiểm tra GPU & đồng bộ mã nguồn
!nvidia-smi

import os
from pathlib import Path

REPO_DIR = Path('/content/skin-cancer-screening-system')
if not REPO_DIR.exists():
    !git clone https://github.com/khoadz2609/skin-cancer-screening-system.git /content/skin-cancer-screening-system
else:
    !git -C /content/skin-cancer-screening-system pull --ff-only

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# 2. Cài đặt dependencies cần thiết trên Colab
!pip -q install -r requirements.txt
!pip -q install kaggle

In [ ]:
# 3. Thiết lập Kaggle Token & tải ISIC 2024 + Stage 1 TabM weights về SSD cục bộ (/content)
import os
from getpass import getpass
from pathlib import Path

if not os.environ.get('KAGGLE_API_TOKEN') and not Path.home().joinpath('.kaggle/kaggle.json').exists():
    token = getpass('Nhập Kaggle API token (KGAT_...): ').strip()
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token

DATA_DIR = Path('/content/isic2024_data')
TABM_DIR = Path('/content/isic_weights/tabm')
DATA_DIR.mkdir(parents=True, exist_ok=True)
TABM_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / 'train-metadata.csv').is_file() or not (DATA_DIR / 'train-image.hdf5').is_file():
    !kaggle competitions download -c isic-2024-challenge -p /content/isic2024_data
    !unzip -q -o /content/isic2024_data/isic-2024-challenge.zip -d /content/isic2024_data

if not any(TABM_DIR.rglob('*.pt')):
    !kaggle datasets download -d khoangdz/isic-2024-tabm-backbone -p /content/isic_weights/tabm --unzip

if not any(Path('backend/backbone_image').glob('*.pt')):
    !kaggle datasets download -d khoangdz/isic-2024-effb0-backbone -p backend/backbone_image --unzip

print('Stage 1 TabM files:', sorted(p.name for p in TABM_DIR.rglob('*.pt')))
print('Stage 2 Image files:', sorted(p.name for p in Path('backend/backbone_image').glob('*.pt')))

In [ ]:
# 4. Chạy kiểm thử unit tests nhanh tại chỗ trước khi train
!python -m pytest backend/mid_fusion_lite/tests/test_bidirectional_fusion.py -k "not end_to_end" -q

In [ ]:
# 5A. Exp 1 (Kiểm chứng thuần kiến trúc Bidirectional Co-Attention + Residual Anchor trên Fold 0, 1)
# Tạo luôn /content/tabular_cache_isic2024 để Exp 2 tái sử dụng tức thời
!python backend/mid_fusion_lite/run_v2_stage3_colab.py \
    --exp arch_only \
    --folds 0,1 \
    --batch-size 64 \
    --valid-batch-size 512 \
    --num-workers 8

In [ ]:
# 5B. Exp 2 (Bật thêm SmoothPAUCLoss + Hard Negative Mining trên Fold 0, 1)
!python backend/mid_fusion_lite/run_v2_stage3_colab.py \
    --exp arch_pauc_hnm \
    --folds 0,1 \
    --batch-size 64 \
    --valid-batch-size 512 \
    --num-workers 8

In [ ]:
# 6. Nếu Fold 0, 1 vượt baseline (0.17083 / 0.17814) -> Chạy Full 5 Folds OOF benchmark
# Đổi --exp sang biến thể tốt nhất giữa arch_only / arch_pauc_hnm / arch_pauc_hnm_tta
!python backend/mid_fusion_lite/run_v2_stage3_colab.py \
    --exp arch_only \
    --folds 0,1,2,3,4 \
    --batch-size 64 \
    --valid-batch-size 512 \
    --num-workers 8

In [ ]:
# 7. So sánh metrics mới với baseline v1 (backend/models/metrics.json)
import json
from pathlib import Path

v1 = json.loads(Path('backend/models/metrics.json').read_text())
print(f"Baseline v1 OOF pAUC: {v1['cv_pauc']:.6f}")
for exp_name in ('arch_only', 'arch_pauc_hnm', 'arch_pauc_hnm_tta'):
    m_path = Path(f'/content/fusion_v2_outputs/{exp_name}/metrics.json')
    if m_path.is_file():
        m = json.loads(m_path.read_text())
        print(f"{exp_name:20s} | OOF pAUC={m['cv_pauc']:.6f} | OOF AUROC={m.get('cv_auc', float('nan')):.6f}")
        for fm in m.get('fold_metrics', []):
            print(f"   Fold {fm['fold']}: pAUC={fm['fold_valid_pauc']:.6f} (v1={v1['fold_metrics'][fm['fold']]['fold_valid_pauc']:.6f})")

In [ ]:
# 8. (Tuỳ chọn) Sao lưu toàn bộ checkpoints, metrics, OOF predictions & tabular cache lên Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')
backup_root = Path('/content/drive/MyDrive/isic2024_stage3_v2_backup')
backup_root.mkdir(parents=True, exist_ok=True)

for src_dir, dst_name in [
    ('/content/fusion_v2_outputs', 'fusion_v2_outputs'),
    ('/content/tabular_cache_isic2024', 'tabular_cache_isic2024'),
]:
    src = Path(src_dir)
    if src.exists():
        shutil.copytree(src, backup_root / dst_name, dirs_exist_ok=True)
        print('Backed up:', src, '->', backup_root / dst_name)